# S2 · The KL-regularized optimum → DPO

Study path stage 2 of 7 · Chapter 11 · Day 18 · **DGX Spark GPU first**

Reinforcement learning from human feedback maximizes reward while charging a cost for moving away
from a frozen reference model. This notebook solves that problem exactly for one prompt with five
possible answers, turns the solution around into a reward defined by the policy itself, and
arrives at the DPO loss. You then train with DPO twice: on the five-answer table, where the right
answer is known, and on Qwen3-0.6B with real human preference pairs. By the end you'll be able to
write `kl`, the optimum `π*`, the DPO loss and its gradient, the masked sequence log-probability
and one DPO training step yourself, and say what a short DPO run does and doesn't establish.

| Part | Question | Discussions in the [learning note](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
|---|---|---|
| A · KL divergence | What does KL(π‖π_ref) measure, and why does its direction matter? | D35 |
| B · The optimum | Which policy maximizes E[R] − β·KL, and what does β do to it? | D35, D21 |
| C · Inverting the optimum | How does a policy define a reward, and why does the partition function drop out? | D21, D32 |
| D · The DPO loss | What does the loss push, and how hard for each pair? | D21, D32 |
| E · Tabular DPO | Does DPO on sampled pairs recover π*? What do the number of pairs and β change? | D14, D34 |
| F · Sequence log-probs (Qwen3) | Which tokens make up log π(y \| x) for a real answer? | D26, D35 |
| G · DPO on Qwen3 | After a few hundred real pairs: held-out accuracy, chosen vs rejected likelihood, KL to the reference | D14, D33, D34 |

RL terms used here (policy, reward, KL penalty, on-policy/offline) are defined in the glossary
[RL terms in the LLM setting](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md#-rl-terms-in-the-llm-setting-glossary-d39).

**How the exercises work.** Each exercise has the same rhythm:

1. **Read the task**, then write your lines where you see `...` in the exercise cell.
2. **Run the check.** It tests your code on small cases and prints ✅ or ❌ with a hint. Nothing
   crashes if you haven't finished; you'll see ⏳ instead.
3. **Compare with the reference**, which follows each check and then uses the result.

Later cells use your version if its check passed, and the reference otherwise
(`mine_or_ref(globals(), …)`), so one stuck exercise never blocks the rest. Predict cells ask a
question first: write your answer down before running the reference.

**Data.** Parts A–E use authored toy distributions (one prompt, three or five answers). Parts F–G
use Anthropic's [HH-RLHF](https://huggingface.co/datasets/Anthropic/hh-rlhf) `helpful-base`
subset (MIT license; Bai et al. 2022): pairs of assistant replies to the same conversation, one
marked better by a human. Only pairs whose two transcripts differ in the last reply are kept.

**Environment.** Kernel `Python (DGX Spark Native)` with CUDA; Qwen3-0.6B and the HH-RLHF files in
the local Hugging Face cache (nothing is downloaded). Checks and references come from the tested
module `dongxi_llms.study_path.s2_kl_optimum_and_dpo`; Qwen3 plumbing from
`dongxi_llms.study_path.qwen`. Run time is about 8 minutes on the Spark, almost all of it Part G;
peak GPU memory is about 22 GiB (two float32 Qwen3 copies, the optimizer state and full-vocabulary float32
logits for 16 sequences).

**Scale and limits.** The toy parts are exact: their numbers are asserted, not estimated. The Qwen3
runs train on 512 pairs for 2 epochs (128 updates), once with β = 0.1 and once with β = 0.5. They
verify the mechanics (masked log-probabilities, a frozen reference, the loss at log 2, the margin
growing) and measure what changed, with standard errors, on 256 held-out pairs and one seed. They
don't produce a better assistant: implicit-reward accuracy on HH-RLHF pairs is not answer quality.

In [ ]:
import copy, inspect, math, random, sys, time
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dongxi_llms").is_dir())
sys.path.insert(0, str(ROOT / "src"))               # the course package, wherever this copy lives
import matplotlib.pyplot as plt
import numpy as np
import torch, torch.nn.functional as F
import transformers
from dongxi_llms.study_path import qwen as Q
from dongxi_llms.study_path import s2_kl_optimum_and_dpo as s2
from dongxi_llms.study_path.common import mine_or_ref

assert torch.cuda.is_available(), "Use the Python (DGX Spark Native) kernel: this notebook is GPU first."
DEVICE = "cuda"
transformers.logging.set_verbosity_error()
random.seed(0); np.random.seed(0); torch.manual_seed(0)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, GRAY, GREEN, RED = "#2563EB", "#EA580C", "#94A3B8", "#16A34A", "#DC2626"
ANSWER_COLORS = [GRAY, BLUE, GREEN, ORANGE, RED]
torch.set_printoptions(precision=4, sci_mode=False)

# Toy: one prompt, five complete answers (float64, CPU: exact and instant)
REF, REWARD, BETA = s2.TOY_REF, s2.TOY_REWARD, s2.TOY_BETA
print("π_ref :", REF.tolist())
print("R     :", REWARD.tolist(), "| β =", BETA)
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

---

# Part A · KL divergence between two answer distributions

Start with a single prompt $x$ and a small finite set of possible answers $y$. A **policy** $\pi$
is then a probability vector over those answers, and so is the frozen **reference policy**
$\pi_{\mathrm{ref}}$ (the model before preference training). We need one number that says how far
$\pi$ has moved from $\pi_{\mathrm{ref}}$. The **Kullback–Leibler divergence** is the average
log-ratio, weighted by the first distribution:

$$\mathrm{KL}(\pi\,\|\,\pi_{\mathrm{ref}}) = \sum_y \pi(y)\,\log\frac{\pi(y)}{\pi_{\mathrm{ref}}(y)}.$$

It is $\ge 0$, equals 0 only when the two distributions are identical, and isn't symmetric. With
D35's example $\pi_{\mathrm{ref}} = (0.5, 0.3, 0.2)$ and $\pi = (0.7, 0.2, 0.1)$, the log-ratios are
$(+0.336, -0.405, -0.693)$ and the weighted sum is $0.0851$. In a language model, $y$ is a whole
answer (or one next token), and the same formula applies.

### Prediction

## Exercise 1 · KL between two categorical distributions

Write `my_kl(p, q)` = KL(p‖q) over the **last** dimension, so a batch `[B, K]` gives `[B]`.
One line. The check uses D35's example, the reverse direction, and a batch of two rows.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_kl(p, q):
    """KL(p‖q) = Σ_y p(y)·log(p(y)/q(y)), summed over the last dimension."""
    return ...

In [ ]:
ex1_ok = s2.check_kl(globals().get("my_kl"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def kl_ref(p, q):
    return torch.special.xlogy(p, p / q).sum(-1)        # xlogy(0, ·) = 0: answers with p(y) = 0 add nothing

s2.check_kl(kl_ref, strict=True)
kl = mine_or_ref(globals(), "my_kl", ex1_ok, kl_ref)

ref3, pi3 = s2.D35_REF, s2.D35_PI
print("log(π/π_ref) per answer:", (pi3 / ref3).log())
print("π-weighted terms        :", pi3 * (pi3 / ref3).log())
print(f"KL(π‖π_ref) = {float(kl(pi3, ref3)):.4f}   KL(π_ref‖π) = {float(kl(ref3, pi3)):.4f}")

### Reference explanation

### Reference answer and mechanism

`(p * (p / q).log()).sum(-1)` is the definition. On D35's example it gives **0.0851**; the reverse
direction KL(π_ref‖π) gives **0.0920**. The two differ because the weights differ: KL(π‖π_ref)
averages the log-ratio over answers *the policy* produces, so it is large when π puts mass where
π_ref has little. RLHF uses this direction because the samples come from π (D35, point 4).

Two edge cases matter later. If π(y) = 0 the term is 0 (the convention 0·log 0 = 0; the reference
uses `torch.special.xlogy` so this doesn't turn into `nan`). If π_ref(y) = 0 while π(y) > 0 the term
is infinite: the policy can never put mass on an answer the reference rules out.

---

# Part B · The KL-regularized objective has a closed-form optimum

RLHF chooses a policy by trading reward against distance from the reference. For one prompt with
reward $R(y)$ per answer, the objective is

$$J(\pi) = \mathbb{E}_{y\sim\pi}[R(y)] - \beta\,\mathrm{KL}(\pi\,\|\,\pi_{\mathrm{ref}})
        = \sum_y \pi(y)R(y) - \beta\sum_y \pi(y)\log\frac{\pi(y)}{\pi_{\mathrm{ref}}(y)}.$$

$\beta > 0$ is the **KL coefficient**: the price, in reward units, of one nat of divergence.
Chapter 11 §11.2 maximizes $J$ over probability vectors with a Lagrange multiplier for
$\sum_y \pi(y) = 1$. Setting the derivative to zero gives $\log(\pi(y)/\pi_{\mathrm{ref}}(y)) =
R(y)/\beta + \text{const}$, so

$$\pi^*(y) = \frac{\pi_{\mathrm{ref}}(y)\,\exp(R(y)/\beta)}{Z},\qquad
  Z = \sum_{y'} \pi_{\mathrm{ref}}(y')\,\exp(R(y')/\beta).$$

$Z$ is the **partition function**: the sum that makes the weights a probability distribution.
Computed as $\mathrm{softmax}(\log\pi_{\mathrm{ref}} + R/\beta)$ it never overflows.

Our toy has five answers. The reference prefers answer 0; the reward prefers answer 3, which the
reference rates at only 0.12:

| answer | 0 | 1 | 2 | 3 | 4 |
|---|---:|---:|---:|---:|---:|
| $\pi_{\mathrm{ref}}$ | 0.40 | 0.25 | 0.15 | 0.12 | 0.08 |
| $R$ | 0.0 | 1.0 | 0.5 | 2.0 | −1.0 |

### Prediction

## Exercise 2 · the optimum π* as one softmax

Write `my_optimal_policy(ref, reward, beta)` = π* over the last dimension. One line, with
`torch.softmax`. The check uses Chapter 11's three-answer fixture (β = 0.5 and β = 1) and adds 50 to
every reward.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_optimal_policy(ref, reward, beta):
    """π*(y) = π_ref(y)·exp(R(y)/β) / Z."""
    return ...

In [ ]:
ex2_ok = s2.check_optimal_policy(globals().get("my_optimal_policy"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def optimal_policy_ref(ref, reward, beta):
    return torch.softmax(ref.log() + reward / beta, -1)

s2.check_optimal_policy(optimal_policy_ref, strict=True)
optimal_policy = mine_or_ref(globals(), "my_optimal_policy", ex2_ok, optimal_policy_ref)

weights = REF * torch.exp(REWARD / BETA)               # unnormalized π_ref·exp(R/β)
Z = weights.sum()
STAR = optimal_policy(REF, REWARD, BETA)
print("π_ref·exp(R/β):", weights)
print(f"Z = {float(Z):.6f}")
print("π*           :", STAR)
torch.testing.assert_close(STAR, weights / Z)

### Reference explanation

### Reference answer and mechanism

`softmax(log π_ref + R/β)` equals `π_ref·exp(R/β) / Z` term by term, because
$\exp(\log\pi_{\mathrm{ref}} + R/\beta) = \pi_{\mathrm{ref}}\exp(R/\beta)$ and softmax divides by
the sum. On the toy, $Z = 2.2430$ and π* = (0.178, 0.303, 0.110, 0.395, 0.013). Answer 3 gains most
(0.12 → 0.395) but doesn't take everything: answer 1 keeps 0.303 because it starts with twice the
reference mass. Adding 50 to every reward multiplies every weight and $Z$ by $e^{50/\beta}$, so π*
is unchanged: only reward *differences* within a prompt matter (Chapter 10's reward gauge).

### Prediction

## Exercise 3 · the objective J(π), and a check that nothing beats π*

Write `my_objective(pi, ref, reward, beta)` = $J(\pi)$, using your `kl` from Exercise 1.
One line. The reference then compares $J(\pi^*)$ with 20,000 random policies.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_objective(pi, ref, reward, beta):
    """J(π) = E_π[R] − β·KL(π‖π_ref), over the last dimension."""
    return ...

In [ ]:
ex3_ok = s2.check_objective(globals().get("my_objective"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def objective_ref(pi, ref, reward, beta):
    return (pi * reward).sum(-1) - beta * kl(pi, ref)

s2.check_objective(objective_ref, strict=True)
objective = mine_or_ref(globals(), "my_objective", ex3_ok, objective_ref)

J_star = objective(STAR, REF, REWARD, BETA)
gen = torch.Generator().manual_seed(0)
print(f"J(π*) = {float(J_star):.6f}   β·log Z = {BETA * float(Z.log()):.6f}")
for scale in (0.01, 0.1, 1.0, 3.0):                  # perturb π* in logit space, small to large
    others = torch.softmax(STAR.log() + scale * torch.randn(5000, 5, generator=gen, dtype=torch.float64), -1)
    J = objective(others, REF, REWARD, BETA)
    assert (J < J_star).all()
    torch.testing.assert_close(J_star - J, BETA * kl(others, STAR))   # the gap is exactly β·KL(π‖π*)
    print(f"perturbation scale {scale:4}: best of 5000 random policies J = {float(J.max()):.6f} "
          f"(gap {float(J_star - J.max()):.2e})")
one_hot = torch.zeros(5, dtype=torch.float64); one_hot[REWARD.argmax()] = 1
print(f"J(π_ref) = {float(objective(REF, REF, REWARD, BETA)):.4f}   "
      f"J(always answer 3) = {float(s2.ref_objective(one_hot, REF, REWARD, BETA)):.4f}")   # xlogy handles 0·log 0

### Reference explanation

### Reference answer and mechanism

$J = \sum_y\pi(y)R(y) - \beta\,\mathrm{KL}(\pi\|\pi_{\mathrm{ref}})$. No random policy beats π*, and
the reference asserts more than that: the gap is **exactly** $\beta\,\mathrm{KL}(\pi\|\pi^*)$.
Substitute $\pi_{\mathrm{ref}}(y)e^{R(y)/\beta} = Z\pi^*(y)$ into $J$:

$$J(\pi) = \beta\sum_y\pi(y)\log\frac{\pi_{\mathrm{ref}}(y)e^{R(y)/\beta}}{\pi(y)}
        = \beta\sum_y\pi(y)\log\frac{Z\,\pi^*(y)}{\pi(y)} = \beta\log Z - \beta\,\mathrm{KL}(\pi\|\pi^*).$$

So $J(\pi^*) = \beta\log Z = 0.8078$, and every other policy is worse by β times its KL to π*. The two
extremes are worse for opposite reasons: π_ref gets no KL cost but only $E[R] = 0.485$; always
answering 3 gets $R = 2$ but pays $\beta\log(1/0.12) = 2.12$ in KL, so $J = -0.12$.

### Prediction

## Predict 1 · what β does to π*

Hold π_ref and R fixed. (a) What does π* become as β → 0, and as β → ∞? (b) Answer 1 has
the second-highest reward and a fairly large reference probability. As β decreases from 100 to
0.01, does π*(answer 1) rise steadily, fall steadily, or do something else?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
betas = torch.logspace(-2, 2, 400, dtype=torch.float64)
P = torch.stack([optimal_policy(REF, REWARD, float(b)) for b in betas])          # [400, 5]
ER = (P * REWARD).sum(-1)
KLs = kl(P, REF.expand_as(P))
k = int(P[:, 1].argmax())
print(f"β = 0.01: π* = {P[0].numpy().round(4)}   β = 100: π* = {P[-1].numpy().round(4)}")
print(f"π*(answer 1) peaks at {float(P[k, 1]):.3f} when β = {float(betas[k]):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for y in range(5):
    axes[0].plot(betas, P[:, y], color=ANSWER_COLORS[y], label=f"answer {y}: π_ref {REF[y]:.2f}, R {REWARD[y]:+.1f}")
axes[0].set_xscale("log"); axes[0].set_xlabel("β (log scale)"); axes[0].set_ylabel("π*(y)")
axes[0].axvline(BETA, color=GRAY, linestyle=":", linewidth=1)
axes[0].set_title("the optimum moves from argmax R (small β) to π_ref (large β)"); axes[0].legend(fontsize=8)
axes[1].plot(betas, ER, color=BLUE, label="E_π*[R]")
axes[1].plot(betas, KLs, color=ORANGE, label="KL(π*‖π_ref), nats")
axes[1].set_xscale("log"); axes[1].set_xlabel("β (log scale)"); axes[1].legend()
axes[1].set_title("reward bought with divergence")
plt.tight_layout(); plt.show()
assert int(P[0].argmax()) == int(REWARD.argmax()) and float(s2.tv(P[-1], REF)) < 0.02

### Reference explanation

### Reference answer and mechanism

(a) As β → 0 the reward term dominates: π* → a one-hot on argmax R (answer 3; at β = 0.01 it holds
all the mass). As β → ∞ the KL term dominates: π* → π_ref (at β = 100 the total-variation distance is
below 0.02). The right panel shows the trade: as β falls, E[R] rises toward 2 and KL(π*‖π_ref) rises
toward log(1/0.12) = 2.12 nats.

(b) Neither: π*(answer 1) **rises then falls**, peaking at 0.31 near β ≈ 1.3. Coming down from large
β, answer 1 gains mass from the low-reward answers 0, 2 and 4. Below β ≈ 1.3 the competition is with
answer 3, whose reward is higher by 1.0: the ratio π*(3)/π*(1) = (0.12/0.25)·e^{1/β} grows without
bound as β → 0. The same rise-then-fall appears in Chapter 11's three-answer figure.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![π* as a function of β, and the reward-KL trade](../figures/chapter-11/study-path-s2_kl_optimum_and_dpo-01.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part C · Inverting the optimum: a reward defined by the policy

Part B went from a reward to a policy. Now run the arrow backwards. Take the log of π* and solve
for $R$:

$$\log\pi^*(y) = \log\pi_{\mathrm{ref}}(y) + \frac{R(y)}{\beta} - \log Z
\quad\Longrightarrow\quad
R(y) = \beta\log\frac{\pi^*(y)}{\pi_{\mathrm{ref}}(y)} + \beta\log Z.$$

For any policy π, define its **implicit reward** $\hat r(y) = \beta\log(\pi(y)/\pi_{\mathrm{ref}}(y))$:
how much π has raised (or lowered) the log-probability of $y$ relative to the reference, in reward
units. For π = π*, $\hat r(y) = R(y) - \beta\log Z$: the true reward minus a constant shared by
every answer to this prompt.

For a real language model, $Z$ is a sum over every possible answer, which we can never compute.
The Bradley–Terry model (Chapter 10) doesn't need it: the probability that answer $i$ beats answer
$j$ depends only on the score **difference**, $P(i\succ j) = \sigma(s_i - s_j)$, and in
$\hat r(i) - \hat r(j)$ the two $\beta\log Z$ terms cancel.

### Prediction

## Exercise 4 · implicit reward, and the partition function cancels

Write two one-liners: `my_implicit_reward(pi, ref, beta)` returns $\hat r$ for every answer, and
`my_pair_prob(scores, i, j)` returns the Bradley–Terry probability that answer `i` beats answer `j`.
The reference then compares all ten pair probabilities computed from $\hat r$ of π* and from $R$.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_implicit_reward(pi, ref, beta):
    return ...                       # β·log(π/π_ref), one value per answer

def my_pair_prob(scores, i, j):
    return ...                       # Bradley–Terry P(answer i ≻ answer j)

In [ ]:
ex4_ok = s2.check_implicit_reward(globals().get("my_implicit_reward"), globals().get("my_pair_prob"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def implicit_reward_ref(pi, ref, beta):
    return beta * (pi.log() - ref.log())

def pair_prob_ref(scores, i, j):
    return torch.sigmoid(scores[i] - scores[j])

s2.check_implicit_reward(implicit_reward_ref, pair_prob_ref, strict=True)
implicit_reward = mine_or_ref(globals(), "my_implicit_reward", ex4_ok, implicit_reward_ref)
pair_prob = mine_or_ref(globals(), "my_pair_prob", ex4_ok, pair_prob_ref)

r_hat = implicit_reward(STAR, REF, BETA)
print("R       :", REWARD)
print("r̂ of π* :", r_hat)
print("R − r̂   :", REWARD - r_hat, f"  (β·log Z = {BETA * float(Z.log()):.4f})")
print("\npair   P from R   P from r̂")
worst = 0.0
for i in range(5):
    for j in range(i + 1, 5):
        a, b = float(pair_prob(REWARD, i, j)), float(pair_prob(r_hat, i, j))
        worst = max(worst, abs(a - b))
        print(f"{i} ≻ {j}   {a:.6f}   {b:.6f}")
print(f"largest difference: {worst:.1e}")
assert worst < 1e-12

### Reference explanation

### Reference answer and mechanism

$\hat r = \beta(\log\pi - \log\pi_{\mathrm{ref}})$ and $P(i\succ j) = \sigma(\hat r_i - \hat r_j)$.
For π*, $R - \hat r = 0.8078 = \beta\log Z$ for **every** answer, and the ten pair probabilities agree
to rounding error (≈ 1e-16). So a policy carries a reward inside it, up to one constant per prompt,
and that constant is exactly what Bradley–Terry ignores. This is the step that removes the reward
model: if we train π so that $\sigma(\hat r(y_w) - \hat r(y_l))$ fits human preferences, we fit the
same likelihood a reward model would, with π itself as the scorer (D21, D32).

---

# Part D · The DPO loss and its gradient

Replace π* by a trainable policy $\pi_\theta$. For a recorded pair (chosen $y_w$, rejected $y_l$),
the **margin** is the difference of implicit rewards,

$$m_\theta = \beta\Big[\big(\log\pi_\theta(y_w) - \log\pi_{\mathrm{ref}}(y_w)\big)
                     - \big(\log\pi_\theta(y_l) - \log\pi_{\mathrm{ref}}(y_l)\big)\Big],$$

and the **DPO loss** is the Bradley–Terry negative log-likelihood of the recorded winner,
averaged over a batch of $B$ pairs:

$$L_{\mathrm{DPO}} = -\frac1B\sum_b \log\sigma(m_b) = \frac1B\sum_b \mathrm{softplus}(-m_b).$$

The inputs are four numbers per pair: the log-probabilities of the two answers under the policy and
under the frozen reference. We store them as `[B, 2]` tensors, column 0 chosen and column 1
rejected. At the start of training $\pi_\theta = \pi_{\mathrm{ref}}$, so every margin is 0 and the
loss is $\log 2 = 0.693$.

### Prediction

## Exercise 5 · the DPO loss

Write the implicit rewards, the margin and the mean loss. Use `F.logsigmoid`: `log(sigmoid(m))`
underflows to −∞ in float32 once m is around −100, and the check tests that.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_dpo_loss(policy_logps, ref_logps, beta):
    """policy_logps, ref_logps: [B, 2], column 0 chosen, 1 rejected. Returns (mean loss, implicit [B, 2])."""
    implicit = ...                     # β·(log π − log π_ref)
    margin = ...                       # chosen minus rejected, [B]
    loss = ...                         # mean of −log σ(margin)
    return loss, implicit

In [ ]:
ex5_ok = s2.check_dpo_loss(globals().get("my_dpo_loss"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def dpo_loss_ref(policy_logps, ref_logps, beta):
    implicit = beta * (policy_logps - ref_logps.detach())      # the reference is a constant
    return -F.logsigmoid(implicit[:, 0] - implicit[:, 1]).mean(), implicit

s2.check_dpo_loss(dpo_loss_ref, strict=True)
dpo_loss = mine_or_ref(globals(), "my_dpo_loss", ex5_ok, dpo_loss_ref)

# three pairs of sequence log-probs (nats), as a language model would give them
pol = torch.tensor([[-10.0, -12.0], [-30.0, -25.0], [-7.0, -7.5]], dtype=torch.float64)
ref_lp = torch.tensor([[-11.0, -11.0], [-28.0, -26.0], [-7.0, -7.0]], dtype=torch.float64)
loss, implicit = dpo_loss(pol, ref_lp, 0.5)
print("implicit rewards [chosen, rejected]:\n", implicit)
print("margins:", implicit[:, 0] - implicit[:, 1], f"| loss {float(loss):.4f} | at π = π_ref: "
      f"{float(dpo_loss(ref_lp, ref_lp, 0.5)[0]):.4f} = log 2")

### Reference explanation

### Reference answer and mechanism

`implicit = beta * (policy_logps - ref_logps)`, `margin = implicit[:, 0] - implicit[:, 1]`,
`loss = -F.logsigmoid(margin).mean()`. In the example (β = 0.5), pair 0 has margin +1.0 (the policy
raised the chosen answer by 1 nat and lowered the rejected one by 1 nat), pair 1 has −1.5 (chosen
lowered by 2 nats, rejected raised by 1), pair 2 has +0.25 (only the rejected answer moved). Only these reference-relative changes enter: pair 1's chosen
answer has a far lower raw log-probability than pair 2's, and that plays no role. The reference
detaches `ref_logps` so no gradient can reach the frozen model, whatever the caller passes.

### Prediction

## Exercise 6 · the gradient, written by hand

Differentiate the mean loss with respect to the policy log-probs. With $\partial(-\log\sigma(m))/\partial m
= -\sigma(-m)$ and $\partial m/\partial\log\pi_\theta(y_w) = \beta$, $\partial m/\partial\log\pi_\theta(y_l) = -\beta$:

$$\frac{\partial L}{\partial \log\pi_\theta(y_w)} = -\frac{\beta\,\sigma(-m)}{B},\qquad
\frac{\partial L}{\partial \log\pi_\theta(y_l)} = +\frac{\beta\,\sigma(-m)}{B}.$$

Write the per-pair weight $w = \sigma(-m)$ and the `[B, 2]` gradient. The check compares with autograd.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_dpo_grads(policy_logps, ref_logps, beta):
    """∂(mean DPO loss)/∂ policy_logps, shape [B, 2], without autograd."""
    B = policy_logps.shape[0]
    m = beta * ((policy_logps[:, 0] - ref_logps[:, 0]) - (policy_logps[:, 1] - ref_logps[:, 1]))
    w = ...                                 # per-pair weight, [B]
    return torch.stack([..., ...], 1)       # chosen column, rejected column

In [ ]:
ex6_ok = s2.check_dpo_grads(globals().get("my_dpo_grads"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s2.check_dpo_grads(s2.ref_dpo_grads, strict=True)
dpo_grads = mine_or_ref(globals(), "my_dpo_grads", ex6_ok, s2.ref_dpo_grads)
print("gradient for the three example pairs [∂L/∂log π(chosen), ∂L/∂log π(rejected)]:\n", dpo_grads(pol, ref_lp, 0.5))

m = torch.linspace(-6, 6, 241)
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(m, F.softplus(-m), color=BLUE, label="loss  −log σ(m)")
ax.plot(m, torch.sigmoid(-m), color=ORANGE, label="gradient weight  σ(−m)")
for mv in (-3.0, 0.0, 3.0):
    w = float(torch.sigmoid(torch.tensor(-mv)))
    ax.scatter([mv], [w], color=ORANGE, zorder=3); ax.annotate(f"{w:.3f}", (mv, w), textcoords="offset points", xytext=(6, 6))
ax.axvline(0, color=GRAY, linewidth=0.8)
ax.set_xlabel("margin m (implicit reward of chosen − rejected)"); ax.legend()
ax.set_title("pairs the policy already orders correctly (m > 0) get a small push")
plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

`w = torch.sigmoid(-m)` and the gradient is `(-β·w/B, +β·w/B)`. Gradient descent therefore raises
$\log\pi_\theta(y_w)$ and lowers $\log\pi_\theta(y_l)$, each by an amount proportional to σ(−m): 0.5 at
m = 0, 0.953 for a pair ordered the wrong way (m = −3), 0.047 for a pair already ordered well
(m = +3). The loss keeps decreasing as m grows (it has no minimum at a finite margin for a hard
label), but the push fades. Two consequences: β multiplies the gradient, so at m = 0 a larger β
moves the policy faster under plain gradient descent with a fixed learning rate (Adam divides by a
running gradient scale, which largely removes this factor); and the per-answer derivatives say nothing about
where the probability mass goes, because in a network both answers share parameters
(Chapter 11 §11.4).

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![DPO loss and gradient weight as functions of the margin](../figures/chapter-11/study-path-s2_kl_optimum_and_dpo-02.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part E · Tabular DPO: does it find π*?

Parts B–D say: if the preference data come from a Bradley–Terry model with reward $R$, and the
policy is free to take any distribution, then the DPO minimizer is π*. Test that directly. The
**tabular policy** is $\pi_\theta = \mathrm{softmax}(\theta)$ with one free logit per answer,
$\theta \in \mathbb{R}^5$, initialized at $\theta = \log\pi_{\mathrm{ref}}$. The data are $N$ pairs:
draw two different answers uniformly, then let $i$ win over $j$ with probability
$\sigma(R(i) - R(j))$. Pairs are stored as two index vectors: `W[n]` (winner) and `L[n]` (loser).

### Prediction

## Exercise 7 · the DPO loss of a tabular policy

Write the three lines: the policy's log-probabilities over the 5 answers, and the `[N, 2]`
policy and reference log-probs of (winner, loser) for every pair. `dpo_loss` (yours from Exercise 5,
if it passed) does the rest.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_tabular_dpo_loss(theta, W, L, ref_logp, beta):
    """theta: [K] logits; W, L: [N] answer indices of winner and loser; ref_logp: [K]. Mean DPO loss."""
    logp = ...                             # log π_θ over the K answers, [K]
    policy = ...                           # [N, 2]: log π_θ(winner), log π_θ(loser)
    ref = ...                              # [N, 2]: log π_ref(winner), log π_ref(loser)
    return dpo_loss(policy, ref, beta)[0]

In [ ]:
ex7_ok = s2.check_tabular_dpo_loss(globals().get("my_tabular_dpo_loss"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def tabular_dpo_loss_ref(theta, W, L, ref_logp, beta):
    logp = theta.log_softmax(-1)
    return dpo_loss(torch.stack([logp[W], logp[L]], 1), torch.stack([ref_logp[W], ref_logp[L]], 1), beta)[0]

s2.check_tabular_dpo_loss(tabular_dpo_loss_ref, strict=True)
tabular_dpo_loss = mine_or_ref(globals(), "my_tabular_dpo_loss", ex7_ok, tabular_dpo_loss_ref)

W, L = s2.sample_pairs(1000, REWARD, seed=0)
print("first pairs (winner ≻ loser):", [f"{int(w)}≻{int(l)}" for w, l in zip(W[:8], L[:8])])
theta = REF.log().clone().requires_grad_()
opt = torch.optim.Adam([theta], lr=0.1)
for step in range(1500):                           # full batch: all 1000 pairs every step
    loss = tabular_dpo_loss(theta, W, L, REF.log(), BETA)
    opt.zero_grad(); loss.backward(); opt.step()
pi_dpo = theta.detach().softmax(-1)
print(f"\nfinal loss {float(loss):.4f}")
print("π_ref :", REF.numpy().round(3)); print("π*    :", STAR.numpy().round(3)); print("π_DPO :", pi_dpo.numpy().round(3))
print(f"TV(π_DPO, π*) = {float(s2.tv(pi_dpo, STAR)):.3f}   TV(π_ref, π*) = {float(s2.tv(REF, STAR)):.3f}")
# the normalizer cancels: unnormalized logits give the same loss
print("loss with logp = θ (no log_softmax):",
      f"{float(dpo_loss(torch.stack([theta[W], theta[L]], 1), torch.stack([REF.log()[W], REF.log()[L]], 1), BETA)[0]):.6f}",
      f"vs {float(tabular_dpo_loss(theta, W, L, REF.log(), BETA)):.6f}")

### Reference explanation

### Reference answer and mechanism

`logp = theta.log_softmax(-1)`, then `torch.stack([logp[W], logp[L]], 1)` and the same for the
reference. Trained on 1,000 sampled pairs, DPO moves the policy from π_ref (TV distance 0.33 from π*)
to within about 0.05 of π*: it found the KL-regularized optimum without ever seeing $R$, $Z$, or a
sample from the policy. The last line shows a detail that holds for any softmax policy: the loss is
the same with or without `log_softmax`, because $\log\pi_\theta(w) - \log\pi_\theta(l) = \theta_w -
\theta_l$. The normalizer cancels within a prompt exactly as $Z$ did in Part C. DPO only constrains
differences between answers that appear in pairs.

For the sweeps below, `s2.fit_tabular_dpo` minimizes the same loss from pair counts
$C[w, l]$ for many datasets at once:

In [ ]:
print(inspect.getsource(s2.fit_tabular_dpo))
pi_counts, _ = s2.fit_tabular_dpo(s2.pair_counts(W, L), REF, BETA)
print("same data through the counts version:", pi_counts.numpy().round(3))
torch.testing.assert_close(pi_counts, pi_dpo, atol=1e-3, rtol=0)

### Prediction

## Predict 2 · where does DPO converge as the number of pairs grows?

Train tabular DPO on N = 10, 30, 100, … 10,000 sampled pairs (8 datasets each). (a) As N grows,
where does π_DPO go, and roughly how fast does its distance to π* shrink? (b) With N = 10, a pair
type such as (3, 4) is often seen once, with 3 winning. What does the DPO loss want the margin of that
pair to be, and what does that do to the policy?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
Ns, seeds = [10, 30, 100, 300, 1000, 3000, 10000], range(8)
counts = torch.stack([s2.pair_counts(*s2.sample_pairs(n, REWARD, seed=100 * n + s)) for n in Ns for s in seeds])
t0 = time.time()
pis, losses = s2.fit_tabular_dpo(counts, REF, BETA)
pis = pis.view(len(Ns), len(seeds), 5)
d = s2.tv(pis, STAR)                                        # [len(Ns), 8]
print(f"{len(counts)} fits in {time.time() - t0:.1f}s")
one_sided = ((counts > 0) & (counts.transpose(1, 2) == 0)).flatten(1).any(1).view(len(Ns), -1).float().mean(1)
for n, row, frac in zip(Ns, d, one_sided):
    print(f"N = {n:5d}: TV(π_DPO, π*) = {float(row.mean()):.3f} ± {float(row.std() / math.sqrt(len(row))):.3f}"
          f"   datasets with a one-sided pair type: {float(frac):.0%}")

# infinite data: expected soft counts q(w ≻ l) = σ(R_w − R_l) for every ordered pair
soft = torch.sigmoid(REWARD[:, None] - REWARD[None, :]) * (1 - torch.eye(5, dtype=torch.float64))
pi_inf, _ = s2.fit_tabular_dpo(soft, REF, BETA, steps=3000)
print(f"infinite data: TV(π_DPO, π*) = {float(s2.tv(pi_inf, STAR)):.1e}")
assert float(s2.tv(pi_inf, STAR)) < 1e-3

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.6), gridspec_kw={"width_ratios": [1.2, 1]})
x = np.arange(5); wd = 0.2
for k, (vec, lab, col) in enumerate([(REF, "π_ref", GRAY), (STAR, "π* (closed form)", BLUE),
                                     (pis[1, 0], "DPO, N = 30", ORANGE), (pis[-1, 0], "DPO, N = 10,000", GREEN)]):
    axes[0].bar(x + (k - 1.5) * wd, vec, wd, color=col, label=lab)
axes[0].set_xticks(x, [f"answer {i}\nR {float(REWARD[i]):+.1f}" for i in range(5)])
axes[0].set_ylabel("probability"); axes[0].legend(fontsize=8); axes[0].set_title("one dataset per N")
mean, se = d.mean(1), d.std(1) / math.sqrt(d.shape[1])
axes[1].errorbar(Ns, mean, yerr=se, color=BLUE, marker="o", capsize=3, label="TV(π_DPO, π*), mean ± SE of 8")
axes[1].plot(Ns, float(mean[3]) * (np.array(Ns) / Ns[3]) ** -0.5, color=GRAY, linestyle="--", label="∝ 1/√N")
axes[1].set_xscale("log"); axes[1].set_yscale("log"); axes[1].set_xlabel("number of pairs N")
axes[1].legend(fontsize=8); axes[1].set_title("DPO converges to π* as N grows")
plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

(a) π_DPO → π*. With expected (infinite-data) counts the fit reaches π* to within 1e-3 TV, as Part C
predicts: the loss is minimized when $\sigma(\hat r_w - \hat r_l)$ equals the true preference
probability $\sigma(R_w - R_l)$ for every pair, i.e. $\hat r = R + \text{const}$, which is π*. With
sampled pairs the distance shrinks roughly like $1/\sqrt N$ (the dashed guide), the rate of any
maximum-likelihood estimate: in the reference run TV ≈ 0.38 at N = 10 and ≈ 0.012 at N = 10,000.

(b) For a pair type seen only one way, the loss $-\log\sigma(m)$ keeps falling as $m \to \infty$, so
DPO pushes that margin as far as the optimizer allows. Nothing in the DPO loss pulls it back: the KL
term of the original objective is only implicit in the parameterization, and it holds back the
solution only when the data contain both outcomes in the right proportions. At N ≤ 30 every dataset
has such a one-sided pair type (from N = 1,000 on, none does), and the fitted policy piles its mass on a few answers (the
orange bars at N = 30 already show it). This is the finite-data failure that later variants such as
IPO were designed to address, and the reason a real DPO run is stopped early or run for one or two
epochs.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Tabular DPO on sampled pairs: the fitted policy and its distance to π* as N grows](../figures/chapter-11/study-path-s2_kl_optimum_and_dpo-03.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

### Prediction

## Predict 3 · β on a fixed dataset: margin vs distance from the reference

Fit tabular DPO on the same 3,000 pairs with β = 0.1, 0.25, 0.5, 1, 2, 4. For each fit, compute
(a) the mean margin $m$ over the training pairs (in reward units) and (b) KL(π_DPO‖π_ref). Which of
the two depends on β, and how?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
bs = [0.1, 0.25, 0.5, 1.0, 2.0, 4.0]
Wb, Lb = s2.sample_pairs(3000, REWARD, seed=7)
Cb = s2.pair_counts(Wb, Lb)
rows = []
for b in bs:
    pi_b, _ = s2.fit_tabular_dpo(Cb, REF, b, steps=2000)
    imp = implicit_reward(pi_b, REF, b)
    margin = float((imp[Wb] - imp[Lb]).mean())
    rows.append((b, margin, float(kl(pi_b, REF)), float(kl(optimal_policy(REF, REWARD, b), REF)),
                 float((pi_b.log() - REF.log())[3] - (pi_b.log() - REF.log())[0])))
    print(f"β = {b:4}: mean margin {margin:.4f} | KL(π_DPO‖π_ref) {rows[-1][2]:.3f} (π*: {rows[-1][3]:.3f})"
          f" | log-ratio gap answer 3 vs 0: {rows[-1][4]:.2f}")
margins = np.array([r[1] for r in rows])
assert margins.max() - margins.min() < 1e-3          # the fitted margin doesn't depend on β

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(bs, margins, color=BLUE, marker="o")
axes[0].set_ylim(0, 1.5 * margins.max()); axes[0].set_xscale("log")
axes[0].set_xlabel("β"); axes[0].set_ylabel("mean margin over training pairs"); axes[0].set_title("margin: the same for every β")
axes[1].plot(bs, [r[2] for r in rows], color=ORANGE, marker="o", label="DPO fit")
axes[1].plot(bs, [r[3] for r in rows], color=GRAY, linestyle="--", label="π* closed form")
axes[1].set_xscale("log"); axes[1].set_xlabel("β"); axes[1].set_ylabel("KL(π‖π_ref), nats")
axes[1].legend(); axes[1].set_title("distance from the reference: shrinks as β grows")
plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

(a) The fitted margin is **the same for every β** (to 1e-3; 1.065 in reward units here). At
the minimum, $\sigma(m_{wl})$ matches the data's win rates, and that condition is stated in reward
units with no β in it. (b) What changes is how far the log-probabilities move to produce that
margin: the log-ratio gap is $m/\beta$ (answer 3 vs answer 0: 0.51 nats at β = 4, 20.4 at β = 0.1),
so KL(π_DPO‖π_ref) grows from 0.02 nats at β = 4 to 2.12 nats at β = 0.1, tracking the closed-form π*. A small β lets the same preference data
move the policy further from the reference.

On a real network trained for a fixed number of steps, β has a second effect from Exercise 6: the
gradient is β·σ(−m), so under plain gradient descent a larger β moves the policy faster at the
start. The converged picture above and that early-training effect are about different regimes
(Chapter 11 §11.4). Part G measures what happens on Qwen3 after 128 Adam updates with β = 0.1 and
β = 0.5.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![β on a fixed dataset: margin is β-independent, KL to the reference is not](../figures/chapter-11/study-path-s2_kl_optimum_and_dpo-04.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part F · Qwen3: the sequence log-probability of an answer

A real answer $y = (y_1, \dots, y_T)$ is a sequence of tokens, and the policy factorizes it:

$$\log\pi_\theta(y\mid x) = \sum_{t=1}^{T}\log\pi_\theta(y_t \mid x, y_{<t}).$$

Every term of DPO's derivation used $\log\pi(y\mid x)$ for a **complete** answer, so the sum runs
over the answer tokens and the end-of-turn token `<|im_end|>` (stopping is part of the answer),
never over prompt tokens, which are context. A causal language model's logits at position $t$
predict the token at position $t+1$. With `ids` of shape `[N, T]` and an `answer_mask` that is 1 at
the answer and `<|im_end|>` positions of `ids`, the log-probability of token `ids[:, t]` is read from
`logits[:, t-1]`.

The data are HH-RLHF pairs. `Q.hh_pairs` parses each transcript into chat messages
(`{"prompt": [...messages], "chosen": str, "rejected": str}`); we keep pairs whose prompt plus
longer answer fits in 192 tokens, which keeps every batch short (and the memory of two float32
models in Part G under 25 GiB). `s2.encode_pairs` lays out each pair
as two rows (chosen, rejected) with the same prompt:

In [ ]:
tok = Q.load_tokenizer()
MAX_LEN = 192

def fits(p):
    n_prompt = len(tok(Q.chat_prompt(tok, p["prompt"]), add_special_tokens=False).input_ids)
    n_answer = max(len(tok(p[k] + "<|im_end|>", add_special_tokens=False).input_ids) for k in ("chosen", "rejected"))
    return n_prompt + n_answer <= MAX_LEN

t0 = time.time()
train_pairs = [p for p in Q.hh_pairs("train")[:3000] if fits(p)][:512]
test_pairs = [p for p in Q.hh_pairs("test")[:600] if fits(p)][:256]
print(f"{len(train_pairs)} training pairs, {len(test_pairs)} held-out pairs ({time.time() - t0:.0f}s)")
ex = test_pairs[0]
print("prompt  :", ex["prompt"][-1]["content"][:200])
print("chosen  :", ex["chosen"][:200])
print("rejected:", ex["rejected"][:200])
print(inspect.getsource(s2.encode_pairs))

### Prediction

## Exercise 8 · the masked sequence log-probability

Write `my_sequence_logps(logits, ids, answer_mask)`: the log-softmax of the logits, the
log-probability of each actual next token (shift by one), and the sum over answer positions only.
Two lines. The check uses random logits for two short sequences; the reference then compares your
function on Qwen3 with `Q.answer_logps` from Day 15 · 04.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_sequence_logps(logits, ids, answer_mask):
    """logits [N, T, V], ids [N, T], answer_mask [N, T] (1 on answer tokens + <|im_end|>). Returns [N]."""
    lp = ...                 # log π(ids[:, t] | ids[:, :t]) for t = 1..T−1, shape [N, T−1]
    return ...               # summed over answer positions only

In [ ]:
ex8_ok = s2.check_sequence_logps(globals().get("my_sequence_logps"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s2.check_sequence_logps(s2.ref_sequence_logps, strict=True)
sequence_logps = mine_or_ref(globals(), "my_sequence_logps", ex8_ok, s2.ref_sequence_logps)

def logps(model, batch):
    """[B, 2] answer log-probs (chosen, rejected) with the sequence_logps chosen above.

    Matrix multiplications run in bfloat16 (autocast); the models keep float32 weights (Part G)."""
    with torch.autocast("cuda", dtype=torch.bfloat16):
        return s2.pair_logps(model, batch, sequence_logps)

print(inspect.getsource(Q.answer_logps))
ref = Q.load_chat_model(dtype=torch.float32); ref.eval(); ref.requires_grad_(False)   # the frozen reference (float32: see Part G)
b = s2.encode_pairs(tok, test_pairs[:4])
row = 0
print("graded tokens of row 0:", repr(tok.decode(b["ids"][row][b["amask"][row].bool()]))[:300])
with torch.no_grad():
    mine = logps(ref, b)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        day15 = Q.answer_logps(ref, tok, test_pairs[:4])
print("this notebook  [chosen, rejected]:\n", mine)
print("Q.answer_logps [chosen, rejected]:\n", day15)
torch.testing.assert_close(mine, day15, atol=0.05, rtol=0)

### Reference explanation

### Reference answer and mechanism

`lp = logits[:, :-1].log_softmax(-1).gather(-1, ids[:, 1:, None]).squeeze(-1)` drops the last logit
(it predicts a token after the sequence) and the first id (nothing predicts it), so `lp[:, t-1]` is
$\log\pi(\text{ids}_t \mid \text{ids}_{<t})$. The mask must shift the same way: `answer_mask[:, 1:]`.
On Qwen3 the result agrees with `Q.answer_logps` (the same layout, computed independently); small
differences come from bf16 kernels on different batch shapes.

Typical values are tens of nats: a 40-token answer at an average 1 nat per token has
$\log\pi \approx -40$. That is why DPO works with log-ratios and differences: the raw numbers depend
mostly on length, the changes from training are small corrections on top.

---

# Part G · DPO on Qwen3-0.6B with real preference pairs

Two copies of Qwen3-0.6B are in memory: the **policy** $\pi_\theta$ (trained) and the frozen
**reference** $\pi_{\mathrm{ref}}$ (the same weights, never updated). One DPO step reads a batch
of 8 recorded pairs, computes the four log-probabilities per pair (two answers × two models), and
takes one optimizer step on the DPO loss. Nothing is sampled: DPO is **offline**, it learns only
from answers in the dataset (D14, D34).

Settings: β = 0.1, AdamW with learning rate 1e-6, gradient clipping at 1.0, 512 training pairs for
2 epochs (128 updates), evaluated on 256 held-out pairs from the HH-RLHF test split. Both models come
from `Q.load_chat_model(dtype=torch.float32)` (the cached Qwen3-0.6B weights on the GPU), and
their forward passes run under bfloat16 autocast (matrix multiplications in bfloat16, weights kept in
float32). Two reasons for float32 weights:

- **The policy needs float32 master weights.** bfloat16 keeps 8 significant bits, so near a typical
  weight value of 0.02 neighbouring bfloat16 numbers are 2⁻¹³ ≈ 1.2e-4 apart, and an AdamW step of
  1e-6 rounds back to the same value. In a test with a bfloat16 policy, one AdamW step at learning
  rate 5e-6 changed only 5% of the weights; with float32 weights, 98% changed. The run prints the
  fraction of one weight matrix that changed after the first step as a check.
- **The reference must compute exactly what the policy computes at step 0.** With a bfloat16
  reference next to a float32 policy, rounding in the non-matrix-multiply operations made the
  step-0 implicit rewards as large as 0.09 instead of 0. With both in float32 they are exactly 0.

Two float32 copies, the AdamW state and float32 logits over the 151,936-token vocabulary for 16
sequences peak at about 22 GiB.

### Prediction

## Exercise 9 · one DPO training step

Fill in the two lines of the step: the reference log-probs (inside `torch.no_grad()`, so no
graph is built through the frozen model) and the loss. The check runs your step on a tiny CPU model
and checks the loss is log 2 at the first step, that the reference has no graph and doesn't change,
and that the margin turns positive.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_dpo_step(policy, ref, batch, opt, beta, logps):
    with torch.no_grad():
        ref_lp = ...                       # [B, 2] reference log-probs, no graph
    policy_lp = logps(policy, batch)       # [B, 2], with a graph back to the policy weights
    loss, implicit = ...                   # the DPO loss (Exercise 5)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    opt.step(); opt.zero_grad()
    return loss.item(), implicit.detach()

In [ ]:
ex9_ok = s2.check_dpo_step(globals().get("my_dpo_step"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def dpo_step_ref(policy, ref, batch, opt, beta, logps):
    with torch.no_grad():
        ref_lp = logps(ref, batch)
    loss, implicit = dpo_loss(logps(policy, batch), ref_lp, beta)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    opt.step(); opt.zero_grad()
    return loss.item(), implicit.detach()

s2.check_dpo_step(dpo_step_ref, strict=True)
dpo_step = mine_or_ref(globals(), "my_dpo_step", ex9_ok, dpo_step_ref)

### Reference explanation

### Reference answer and mechanism

`ref_lp = logps(ref, batch)` inside `no_grad`, then `loss, implicit = dpo_loss(policy_lp, ref_lp, beta)`.
Without `no_grad` the reference forward would build a graph and keep all its activations until the
backward pass (on Qwen3 that is gigabytes); with `ref.requires_grad_(False)` it wouldn't change the
result, but it would waste the memory. The order matters: the policy is the first argument.
Because the reference never changes, its log-probs could also be computed once for the whole
dataset before training; we recompute them per batch to keep the step self-contained.

### Prediction

## Exercise 10 · implicit-reward accuracy with a standard error

The evaluation metric is the fraction of held-out pairs whose chosen answer has the higher
implicit reward. At the first step every margin is exactly 0, so count a tie as ½ (an untrained
policy then scores exactly 50%). Also return the binomial standard error $\sqrt{a(1-a)/n}$.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_pair_accuracy(implicit):
    """implicit [n, 2] → (accuracy with ties counted ½, its standard error)."""
    m = implicit[:, 0] - implicit[:, 1]
    acc = ...                              # mean of 1 (m > 0), ½ (m == 0), 0 (m < 0)
    se = ...                               # sqrt(acc·(1 − acc) / n)
    return float(acc), float(se)

In [ ]:
ex10_ok = s2.check_pair_accuracy(globals().get("my_pair_accuracy"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s2.check_pair_accuracy(s2.ref_pair_accuracy, strict=True)
pair_accuracy = mine_or_ref(globals(), "my_pair_accuracy", ex10_ok, s2.ref_pair_accuracy)

BETA_Q, LR, BS, EPOCHS = 0.1, 1e-6, 8, 2
policy = Q.load_chat_model(dtype=torch.float32)      # the trainable copy: same weights as ref, float32 master copy
train_batches = [s2.encode_pairs(tok, train_pairs[i:i + BS]) for i in range(0, len(train_pairs), BS)]
test_batches = [s2.encode_pairs(tok, test_pairs[i:i + BS]) for i in range(0, len(test_pairs), BS)]

@torch.no_grad()
def all_logps(model, batches):
    model.eval(); out = torch.cat([logps(model, b) for b in batches]); return out

t0 = time.time()
ref_test, ref_train = all_logps(ref, test_batches), all_logps(ref, train_batches[:32])   # computed once
print(f"reference log-probs for {len(ref_test)} held-out and {len(ref_train)} training pairs ({time.time() - t0:.0f}s)")
acc0, se0 = pair_accuracy(BETA_Q * (all_logps(policy, test_batches) - ref_test))
print(f"step 0 held-out implicit-reward accuracy: {acc0:.1%} ± {se0:.1%} (every margin is 0)")
raw, raw_se = pair_accuracy(ref_test)
print(f"for comparison, the reference's raw log π(chosen) > log π(rejected): {raw:.1%} ± {raw_se:.1%}")
assert acc0 == 0.5

### Reference explanation

### Reference answer and mechanism

`acc = ((m > 0).float() + 0.5 * (m == 0).float()).mean()` and `se = sqrt(acc·(1 − acc)/n)`. Before
training the accuracy is exactly 50%, not "about" 50%: policy and reference are the same weights, so
every implicit reward is exactly 0. With n = 256 the standard error of an accuracy near 50% is
about 3.1 points, so a difference smaller than about 6 points (two SE) is not distinguishable from
noise.

The second number is a different quantity: whether the base model already gives the chosen answer
a higher **raw** log-probability. It is well below 50% (39% ± 3% in the reference run). Length is
one reason: in these 256 held-out pairs the chosen answer is longer in 61% of them (45 vs 32 tokens on
average), and every extra token adds a negative log-probability term. This is why DPO compares each
answer with its own reference log-probability instead of comparing the two answers directly.

### Prediction

## Predict 4 · chosen and rejected likelihoods during DPO

DPO raises the margin $\hat r(y_w) - \hat r(y_l)$. Over 128 updates on real pairs, which do you
expect on the held-out pairs: (a) chosen implicit reward up, rejected down; (b) both up, chosen more;
(c) both down, rejected more? Will the training pairs show the same pattern? And will the held-out
accuracy rise as much as the training accuracy?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
def run_dpo(policy, beta, eval_every=32):
    """EPOCHS passes over train_batches with dpo_step; evaluates on held-out and 256 training pairs."""
    opt = torch.optim.AdamW(policy.parameters(), lr=LR, weight_decay=0.0)
    log, evals, step, t0 = [], [], 0, time.time()

    def evaluate():
        imp_te = beta * (all_logps(policy, test_batches) - ref_test)
        imp_tr = beta * (all_logps(policy, train_batches[:32]) - ref_train)
        policy.train()
        evals.append({"step": step, "test": pair_accuracy(imp_te), "train": pair_accuracy(imp_tr),
                      "test_chosen": float(imp_te[:, 0].mean()), "test_rejected": float(imp_te[:, 1].mean()),
                      "imp_te": imp_te.cpu(), "imp_tr": imp_tr.cpu()})
        e = evals[-1]
        print(f"step {step:3d} | held-out acc {e['test'][0]:.1%} ± {e['test'][1]:.1%} | train acc {e['train'][0]:.1%} "
              f"| held-out mean r̂: chosen {e['test_chosen']:+.3f}, rejected {e['test_rejected']:+.3f} | {time.time() - t0:.0f}s")

    if eval_every:
        evaluate()
    policy.train()
    for epoch in range(EPOCHS):
        for k in torch.randperm(len(train_batches), generator=torch.Generator().manual_seed(epoch)).tolist():
            probe = policy.model.layers[0].mlp.down_proj.weight
            before = probe.detach().clone() if step == 0 else None
            loss, implicit = dpo_step(policy, ref, train_batches[k], opt, beta, logps)
            if step == 0:
                print(f"first step changed {float((probe != before).float().mean()):.1%} of the "
                      f"{probe.numel():,} weights in layer 0's MLP down-projection ({probe.dtype})")
            step += 1
            log.append({"loss": loss, "margin": float((implicit[:, 0] - implicit[:, 1]).mean()),
                        "chosen": float(implicit[:, 0].mean()), "rejected": float(implicit[:, 1].mean())})
            if (eval_every and step % eval_every == 0) or step == EPOCHS * len(train_batches):
                if not evals or evals[-1]["step"] != step:
                    evaluate()
    return log, evals

def describe(evals, beta):
    last = evals[-1]
    for split in ("imp_te", "imp_tr"):
        imp = last[split].numpy(); n = len(imp)
        name = "held-out" if split == "imp_te" else "training"
        d = imp[:, 0] - imp[:, 1]
        print(f"{name:8s} pairs: margin {d.mean():+.3f} ± {d.std() / math.sqrt(n):.3f} | "
              f"chosen r̂ {imp[:, 0].mean():+.3f} ± {imp[:, 0].std() / math.sqrt(n):.3f} "
              f"(log-prob {imp[:, 0].mean() / beta:+.2f} nats) | "
              f"rejected r̂ {imp[:, 1].mean():+.3f} ± {imp[:, 1].std() / math.sqrt(n):.3f} "
              f"(log-prob {imp[:, 1].mean() / beta:+.2f} nats)")

torch.cuda.reset_peak_memory_stats(); t0 = time.time()
log, evals = run_dpo(policy, BETA_Q)
train_time, peak = time.time() - t0, Q.gib(torch.cuda.max_memory_allocated())
print(f"\ntraining + evaluation {train_time:.0f}s | peak GPU memory {peak:.1f} GiB")
describe(evals, BETA_Q)
assert abs(log[0]["loss"] - math.log(2)) < 1e-3

### Reference explanation

### Reference answer and mechanism

The loss can't tell (a), (b) and (c) apart, because it depends only on the difference; only the two
curves can. In the reference run the answer changed during training. After 64 updates the held-out
pairs showed **(b)**: both implicit rewards up (chosen +0.46, rejected +0.27). After 128 updates they
showed **(a)**: chosen +0.20 ± 0.04 and rejected −0.20 ± 0.04 in reward units, i.e. +2.0 and −2.0 nats
of log-probability. The training pairs ended in (a) as well, more strongly: chosen +0.70 ± 0.05 (+7.0
nats), rejected −0.77 ± 0.04 (−7.7 nats). The right panel shows the turn: the rejected curve, on
training batches and held-out pairs, rises with the chosen one during the first epoch and falls
during the second.

Neither direction is guaranteed. Other runs and datasets often show (c), both falling with the
rejected one falling faster: **likelihood displacement**, where probability mass leaves both
compared answers for continuations that appear in no pair (Chapter 11 §11.4's three-answer
counterexample). This run doesn't show (c). In a test run with a five times larger learning rate
(5e-6) it did: after 128 updates both held-out implicit rewards were negative (chosen −1.8, rejected
−2.8), training accuracy was 100%, and the sampled KL to the reference was about 23 nats.

Training accuracy rises much faster than held-out accuracy: 97.3% vs 67.2% after 128 updates. With
512 pairs seen twice, the policy partly memorizes its pairs, the finite-data effect from Predict 2;
the training loss drops sharply after step 64, where the second epoch starts repeating the same
pairs. The held-out accuracy reached 68% after the first 32 updates and then stayed within noise of
that level. The held-out gain, 67.2% ± 2.9% against exactly 50% at step 0, is about six standard
errors: real for this split, measured on one seed and 256 pairs.

In [ ]:
def smooth(x, k=8):
    return np.convolve(x, np.ones(k) / k, mode="valid")

steps = np.arange(1, len(log) + 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
axes[0].plot(steps, [r["loss"] for r in log], color=GRAY, alpha=0.5, label="per batch")
axes[0].plot(steps[7:], smooth([r["loss"] for r in log]), color=BLUE, label="8-step mean")
axes[0].axhline(math.log(2), color=RED, linestyle=":", label="log 2"); axes[0].set_xlabel("DPO step")
axes[0].set_title("training loss (β = 0.1)"); axes[0].legend(fontsize=8)
es = [e["step"] for e in evals]
for key, col, lab in (("test", BLUE, "held-out (256 pairs)"), ("train", ORANGE, "training (256 pairs)")):
    a = np.array([e[key][0] for e in evals]); s = np.array([e[key][1] for e in evals])
    axes[1].errorbar(es, a, yerr=s, color=col, marker="o", capsize=3, label=lab)
axes[1].axhline(0.5, color=GRAY, linestyle=":"); axes[1].set_xlabel("DPO step")
axes[1].set_title("implicit-reward accuracy ± SE"); axes[1].legend(fontsize=8)
axes[2].plot(steps[7:], smooth([r["chosen"] for r in log]), color=GREEN, label="chosen (training batch)")
axes[2].plot(steps[7:], smooth([r["rejected"] for r in log]), color=RED, label="rejected (training batch)")
axes[2].plot(es, [e["test_chosen"] for e in evals], color=GREEN, marker="o", linestyle="--", label="chosen (held-out)")
axes[2].plot(es, [e["test_rejected"] for e in evals], color=RED, marker="o", linestyle="--", label="rejected (held-out)")
axes[2].axhline(0, color=GRAY, linewidth=0.8); axes[2].set_xlabel("DPO step")
axes[2].set_title("mean implicit reward β·log(π/π_ref)"); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![DPO on Qwen3: loss, implicit-reward accuracy, and chosen/rejected implicit rewards](../figures/chapter-11/study-path-s2_kl_optimum_and_dpo-05.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

How far did the policy move? The implicit rewards above are measured on the dataset's answers,
which the policy didn't write. The KL term of the original objective is an expectation over answers
**sampled from the policy itself**: $\mathrm{KL}(\pi_\theta\|\pi_{\mathrm{ref}}) =
\mathbb{E}_{y\sim\pi_\theta}[\log\pi_\theta(y\mid x) - \log\pi_{\mathrm{ref}}(y\mid x)]$. We estimate it
by sampling one answer per held-out prompt at temperature 1 with `Q.rollout`, scoring every sampled
token under both models with `Q.token_logps`, and averaging the per-answer sum of log-ratios (the
estimator **k1** of D35).

In [ ]:
print(inspect.getsource(Q.rollout))
print(inspect.getsource(Q.token_logps))

### Prediction

## Exercise 11 · a sampled estimate of KL(π_θ‖π_ref)

Given per-token log-probs `logp_pi`, `logp_ref` (`[n, T]`) of n sampled answers and their response
mask (1 up to and including `<|im_end|>`, 0 on padding), compute the per-answer k1 value
$\sum_t(\log\pi_\theta - \log\pi_{\mathrm{ref}})$, its mean, and the standard error of the mean.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_kl_estimate(logp_pi, logp_ref, mask):
    """→ (mean KL estimate, its standard error, per-answer values [n])."""
    k1 = ...                                    # Σ over response tokens of the log-ratio, [n]
    return float(k1.mean()), float(...), k1     # standard error of the mean

In [ ]:
ex11_ok = s2.check_kl_estimate(globals().get("my_kl_estimate"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s2.check_kl_estimate(s2.ref_kl_estimate, strict=True)
kl_estimate = mine_or_ref(globals(), "my_kl_estimate", ex11_ok, s2.ref_kl_estimate)
KL_PROMPTS = [p["prompt"] for p in test_pairs[:96]]

def sampled_kl(model, seed=1):
    model.eval(); torch.manual_seed(seed)
    with torch.autocast("cuda", dtype=torch.bfloat16), torch.no_grad():
        ro = Q.rollout(model, tok, KL_PROMPTS, max_new=96)
        lp_pi, lp_ref = Q.token_logps(model, ro), Q.token_logps(ref, ro)
    return kl_estimate(lp_pi, lp_ref, ro["mask"]), ro

t0 = time.time()
(kl_mean, kl_se, k1), ro = sampled_kl(policy)
print(f"KL(π_θ‖π_ref) ≈ {kl_mean:.3f} ± {kl_se:.3f} nats per answer "
      f"({len(k1)} samples, {float(ro['mask'].sum(1).mean()):.0f} tokens on average, "
      f"{float(ro['truncated'].float().mean()):.0%} hit the 96-token limit, {time.time() - t0:.0f}s)")
print(f"answers with a negative k1 value: {float((k1 < 0).float().mean()):.0%}")
print("\nsample:", Q.response_texts(tok, ro)[0][:300])

### Reference explanation

### Reference answer and mechanism

`k1 = ((logp_pi - logp_ref) * mask).sum(1)`, then its mean and `k1.std() / sqrt(n)`. The mask
matters: tokens after `<|im_end|>` are padding the model never produced. Individual k1 values can be
negative (an answer the reference happens to like more); only the mean estimates the KL, and only
because the answers are sampled from π_θ at temperature 1 (D35, point 4). Answers cut off at 96
tokens (most of them here) contribute their first 96 tokens only, so this estimates the KL of the
first 96 tokens of an answer.

In the reference run the estimate is 1.61 ± 0.18 nats per answer (68% of the 96 samples hit the
96-token limit; 21% of the individual k1 values are negative): after 128 updates the policy's own
answers are measurably, but not drastically, less likely under the reference, about 0.02 nats per
token. Compare the toy, where the converged DPO policy at β = 0.1 sat 2.1 nats away from a
five-answer reference.

### Same data, β = 0.5

Predict 3 showed that at convergence the fitted margin doesn't depend on β, while the distance from
the reference shrinks as β grows. A 128-update run is far from convergence, so measure it: a fresh
policy from the same weights, the same 512 pairs, order, learning rate and number of updates, with
β = 0.5 instead of 0.1. Only the final evaluation and the sampled KL are computed.

In [ ]:
results = {BETA_Q: {"evals": evals, "kl": (kl_mean, kl_se)}}
del policy; Q.free()
policy = Q.load_chat_model(dtype=torch.float32)
t0 = time.time()
log_b, evals_b = run_dpo(policy, 0.5, eval_every=None)
(kl_b, kl_b_se, _), _ = sampled_kl(policy)
results[0.5] = {"evals": evals_b, "kl": (kl_b, kl_b_se)}
print(f"β = 0.5 run: {time.time() - t0:.0f}s")
describe(evals_b, 0.5)
print("\n  β  | held-out acc   | held-out margin (reward units) | log-ratio gap (nats) | sampled KL (nats)")
for b, r in results.items():
    e = r["evals"][-1]; d = (e["imp_te"][:, 0] - e["imp_te"][:, 1]).numpy()
    print(f" {b:3} | {e['test'][0]:.1%} ± {e['test'][1]:.1%} | {d.mean():+.3f} ± {d.std() / math.sqrt(len(d)):.3f}"
          f"               | {d.mean() / b:+.2f}               | {r['kl'][0]:.2f} ± {r['kl'][1]:.2f}")
del policy; Q.free()

**Reading the comparison.** In the reference run the two β values gave the same held-out accuracy
within noise (67.2% ± 2.9% vs 69.5% ± 2.9%). The margins differ in the way Predict 3 describes. In
reward units the β = 0.5 margin is larger (+0.89 vs +0.40), but in log-probability it is smaller: the
held-out log-ratio gap is 1.79 nats at β = 0.5 against 4.00 nats at β = 0.1, and the sampled KL is
0.82 ± 0.11 vs 1.61 ± 0.18 nats. So the larger β moved the policy less from the reference, as at
convergence, even though Exercise 6's gradient is five times larger. Two mechanisms explain this.
AdamW rescales each parameter's update by a running gradient magnitude, which removes most of β's
scale factor. And with β = 0.5 each pair reaches a given margin with a smaller log-ratio change, so
σ(−m) falls sooner and the push on already well-ordered pairs fades sooner (training accuracy
reached 100%, against 97.3% at β = 0.1). One seed per β, with matched data order: the KL difference
(0.79 ± 0.21) is about 3.7 combined standard errors, and the accuracy difference is noise.

---

# Summary

| Part | You built | What it showed (reference run) |
|---|---|---|
| A | `kl(p, q)` | KL(π‖π_ref) = 0.0851 on D35's example, 0.0920 the other way: the direction matters |
| B | `optimal_policy`, `objective` | π* = softmax(log π_ref + R/β); J(π) = β log Z − β KL(π‖π*), so π* is the unique maximum; β → 0 gives argmax R, β → ∞ gives π_ref |
| C | `implicit_reward`, `pair_prob` | β log(π*/π_ref) = R − β log Z; every Bradley–Terry pair probability is unchanged, so Z is never needed |
| D | `dpo_loss`, `dpo_grads` | loss starts at log 2; each pair is pushed with weight σ(−m) |
| E | `tabular_dpo_loss` | DPO on sampled pairs → π* as N grows (≈ 1/√N); one-sided pairs at small N overshoot; the fitted margin doesn't depend on β, the KL to π_ref does |
| F | `sequence_logps` | answer + `<|im_end|>` tokens only, shifted by one; matches `Q.answer_logps` |
| G | `dpo_step`, `pair_accuracy`, `kl_estimate` | DPO on 512 HH pairs (float32 master weights), β = 0.1: held-out accuracy 50% → 67.2% ± 2.9%, training 97.3%; held-out chosen implicit reward up (+0.20), rejected down (−0.20) after both rose in epoch 1; sampled KL 1.61 ± 0.18 nats. β = 0.5: same accuracy within noise, smaller KL (0.82 ± 0.11) |

**Evidence.** The toy results are exact and asserted: the closed-form optimum, the β log Z identity,
the cancellation of Z, the infinite-data DPO fit equal to π*, and the β-independence of the fitted
margin. The Qwen3 run verifies the mechanics on a real model: the masked sequence log-probability,
a frozen reference with no graph, a loss that starts at exactly log 2 and falls, and an implicit-reward
accuracy of exactly 50% at step 0. Its measured effects (held-out implicit-reward accuracy, the
direction in which chosen and rejected likelihoods moved, the sampled KL, the β comparison) come from
one seed per β, 512 training pairs and 256 held-out pairs; they are reported with standard errors.
The held-out accuracy gain is well outside noise for this split; the direction of the likelihood
movement is a property of this run, not of DPO. Implicit-reward accuracy measures agreement with
HH-RLHF labels, not the quality of generated answers. Mechanics verified, no capability claim.

**Next stage.** [S3 · REINFORCE → baselines → PPO](s3_policy_gradient_to_ppo.ipynb) returns to the
KL-regularized objective of Part B and optimizes it the other way: **online**, by sampling answers
from the policy and following the policy gradient, with a reward model as the scorer.